# Smart Agriculture ML Model Training
# शेतीसेतु AI मॉडेल प्रशिक्षण

This notebook trains three ML models:
1. **Yield Prediction** - XGBoost Regressor
2. **Risk Assessment** - Gradient Boosting Classifier
3. **Loss Probability** - Random Forest Classifier

In [ ]:
# Install required packages
!pip install scikit-learn xgboost pandas numpy matplotlib seaborn joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report, confusion_matrix
from xgboost import XGBRegressor
import joblib
import os

# Create directories
os.makedirs('../data/models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Libraries loaded successfully!')

## 1. Generate Training Data

We'll generate synthetic data based on Maharashtra agriculture patterns.
In production, replace this with real data from data.gov.in

In [ ]:
# Maharashtra crop base yields (quintals/hectare)
BASE_YIELDS = {
    'rice': 22, 'wheat': 28, 'jowar': 12, 'bajra': 10, 'maize': 25,
    'sugarcane': 800, 'cotton': 4, 'soybean': 15, 'groundnut': 12,
    'onion': 180, 'grapes': 120, 'pomegranate': 100, 'orange': 80
}

# District drought-prone scores (Marathwada region higher)
DROUGHT_SCORES = {
    'pune': 0.4, 'nashik': 0.5, 'aurangabad': 0.8, 'solapur': 0.9,
    'kolhapur': 0.2, 'nagpur': 0.4, 'beed': 0.85, 'latur': 0.8
}

print('Configuration loaded!')

In [ ]:
def generate_yield_data(n_samples=5000):
    """Generate synthetic yield data"""
    np.random.seed(42)
    
    districts = np.random.randint(1, 37, n_samples)
    crops = np.random.randint(1, 29, n_samples)
    seasons = np.random.choice([1, 2, 3, 4], n_samples, p=[0.4, 0.35, 0.15, 0.1])
    areas = np.random.uniform(0.5, 20, n_samples)
    irrigation = np.random.choice([0, 1, 2, 3, 4, 5, 6], n_samples, 
                                  p=[0.35, 0.15, 0.15, 0.15, 0.1, 0.05, 0.05])
    sowing_months = np.random.randint(1, 13, n_samples)
    historical_avg = np.random.uniform(15, 40, n_samples)
    rainfall_dev = np.random.uniform(-50, 50, n_samples)
    
    # Generate yields based on factors
    base_yields = {i: np.random.uniform(10, 200) for i in range(1, 29)}
    base_yields.update({6: 800, 10: 180, 11: 120})  # sugarcane, onion, grapes
    
    yields = []
    for i in range(n_samples):
        base = base_yields.get(crops[i], 20)
        irr_mult = [0.7, 1.1, 1.0, 1.05, 1.2, 1.15, 1.1][irrigation[i]]
        rain_factor = 1 + (rainfall_dev[i] / 200)
        noise = np.random.normal(1, 0.15)
        yields.append(max(0, base * irr_mult * rain_factor * noise))
    
    return pd.DataFrame({
        'district_code': districts, 'crop_code': crops, 'season_code': seasons,
        'area_hectares': areas, 'irrigation_type': irrigation,
        'sowing_month': sowing_months, 'historical_yield_avg': historical_avg,
        'rainfall_deviation': rainfall_dev, 'yield': yields
    })

yield_df = generate_yield_data()
print(f'Generated {len(yield_df)} yield samples')
yield_df.head()

In [ ]:
# Visualize yield distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Yield distribution
axes[0, 0].hist(yield_df['yield'], bins=50, edgecolor='black')
axes[0, 0].set_title('Yield Distribution')
axes[0, 0].set_xlabel('Yield (quintals/hectare)')

# Yield by irrigation type
yield_df.boxplot(column='yield', by='irrigation_type', ax=axes[0, 1])
axes[0, 1].set_title('Yield by Irrigation Type')
axes[0, 1].set_xlabel('Irrigation Type (0=Rainfed)')

# Yield by season
yield_df.boxplot(column='yield', by='season_code', ax=axes[1, 0])
axes[1, 0].set_title('Yield by Season')
axes[1, 0].set_xlabel('Season (1=Kharif, 2=Rabi)')

# Correlation heatmap
corr = yield_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', ax=axes[1, 1], cmap='coolwarm')
axes[1, 1].set_title('Feature Correlation')

plt.tight_layout()
plt.savefig('../data/processed/yield_analysis.png', dpi=150)
plt.show()

## 2. Train Yield Prediction Model

In [ ]:
# Prepare data
X_yield = yield_df.drop('yield', axis=1)
y_yield = yield_df['yield']

X_train, X_test, y_train, y_test = train_test_split(X_yield, y_yield, test_size=0.2, random_state=42)

# Scale features
yield_scaler = StandardScaler()
X_train_scaled = yield_scaler.fit_transform(X_train)
X_test_scaled = yield_scaler.transform(X_test)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

In [ ]:
# Train XGBoost model
yield_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

yield_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = yield_model.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R² Score: {r2:.4f}')
print(f'RMSE: {rmse:.2f} quintals/hectare')

In [ ]:
# Save yield model
joblib.dump(yield_model, '../data/models/yield_model.joblib')
joblib.dump(yield_scaler, '../data/models/yield_model_scaler.joblib')
yield_df.to_csv('../data/processed/yield_training_data.csv', index=False)
print('Yield model saved!')

## 3. Train Risk Assessment Model

In [ ]:
def generate_risk_data(n_samples=5000):
    np.random.seed(43)
    
    districts = np.random.randint(1, 37, n_samples)
    crops = np.random.randint(1, 29, n_samples)
    seasons = np.random.choice([1, 2, 3, 4], n_samples, p=[0.4, 0.35, 0.15, 0.1])
    irrigation = np.random.choice([0, 1, 2, 3, 4, 5, 6], n_samples, p=[0.35, 0.15, 0.15, 0.15, 0.1, 0.05, 0.05])
    areas = np.random.uniform(0.5, 20, n_samples)
    historical_loss = np.random.uniform(0, 0.5, n_samples)
    drought_score = np.random.uniform(0, 1, n_samples)
    flood_score = np.random.uniform(0, 1, n_samples)
    pest_score = np.random.uniform(0, 1, n_samples)
    
    # Calculate risk level (0=low, 1=moderate, 2=high, 3=very_high)
    risk_scores = (
        (irrigation == 0).astype(float) * 25 +
        drought_score * 20 + flood_score * 15 +
        historical_loss * 30 + pest_score * 10 +
        np.random.uniform(0, 20, n_samples)
    )
    risk_levels = np.digitize(risk_scores, [30, 50, 70])
    
    return pd.DataFrame({
        'district_code': districts, 'crop_code': crops, 'season_code': seasons,
        'irrigation_type': irrigation, 'area_hectares': areas,
        'historical_loss_rate': historical_loss, 'drought_prone_score': drought_score,
        'flood_prone_score': flood_score, 'pest_history_score': pest_score,
        'risk_level': risk_levels
    })

risk_df = generate_risk_data()
print(f'Generated {len(risk_df)} risk samples')
print(f'Risk level distribution:\n{risk_df["risk_level"].value_counts().sort_index()}')

In [ ]:
# Train risk model
X_risk = risk_df.drop('risk_level', axis=1)
y_risk = risk_df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X_risk, y_risk, test_size=0.2, random_state=42)

risk_scaler = StandardScaler()
X_train_scaled = risk_scaler.fit_transform(X_train)
X_test_scaled = risk_scaler.transform(X_test)

risk_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
risk_model.fit(X_train_scaled, y_train)

y_pred = risk_model.predict(X_test_scaled)
print(f'Accuracy: {risk_model.score(X_test_scaled, y_test):.4f}')
print(f'\nClassification Report:\n{classification_report(y_test, y_pred)}')

In [ ]:
# Save risk model
joblib.dump(risk_model, '../data/models/risk_model.joblib')
joblib.dump(risk_scaler, '../data/models/risk_model_scaler.joblib')
risk_df.to_csv('../data/processed/risk_training_data.csv', index=False)
print('Risk model saved!')

## 4. Train Loss Probability Model

In [ ]:
def generate_loss_data(n_samples=5000):
    np.random.seed(44)
    
    districts = np.random.randint(1, 37, n_samples)
    crops = np.random.randint(1, 29, n_samples)
    loss_types = np.random.randint(0, 9, n_samples)
    growth_stages = np.random.randint(0, 4, n_samples)
    areas = np.random.uniform(0.5, 20, n_samples)
    rainfall = np.random.uniform(0, 300, n_samples)
    temp_dev = np.random.uniform(-10, 10, n_samples)
    days_sowing = np.random.randint(0, 200, n_samples)
    
    # Calculate loss probability
    loss_prob = (
        (loss_types < 3).astype(float) * 0.2 +
        (growth_stages == 2).astype(float) * 0.15 +
        (rainfall < 20).astype(float) * 0.2 +
        (rainfall > 200).astype(float) * 0.15 +
        (np.abs(temp_dev) > 5).astype(float) * 0.1 +
        np.random.uniform(0, 0.3, n_samples)
    )
    loss_occurred = (loss_prob > 0.5).astype(int)
    
    return pd.DataFrame({
        'district_code': districts, 'crop_code': crops,
        'loss_type_code': loss_types, 'growth_stage': growth_stages,
        'area_hectares': areas, 'rainfall_current_month': rainfall,
        'temperature_deviation': temp_dev, 'days_since_sowing': days_sowing,
        'loss_occurred': loss_occurred
    })

loss_df = generate_loss_data()
print(f'Generated {len(loss_df)} loss samples')
print(f'Loss distribution: {loss_df["loss_occurred"].value_counts().to_dict()}')

In [ ]:
# Train loss model
X_loss = loss_df.drop('loss_occurred', axis=1)
y_loss = loss_df['loss_occurred']

X_train, X_test, y_train, y_test = train_test_split(X_loss, y_loss, test_size=0.2, random_state=42)

loss_scaler = StandardScaler()
X_train_scaled = loss_scaler.fit_transform(X_train)
X_test_scaled = loss_scaler.transform(X_test)

loss_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42
)
loss_model.fit(X_train_scaled, y_train)

y_pred = loss_model.predict(X_test_scaled)
print(f'Accuracy: {loss_model.score(X_test_scaled, y_test):.4f}')
print(f'\nClassification Report:\n{classification_report(y_test, y_pred)}')

In [ ]:
# Save loss model
joblib.dump(loss_model, '../data/models/loss_model.joblib')
joblib.dump(loss_scaler, '../data/models/loss_model_scaler.joblib')
loss_df.to_csv('../data/processed/loss_training_data.csv', index=False)
print('Loss model saved!')

## 5. Summary

All three models have been trained and saved:
- `yield_model.joblib` - Crop yield prediction
- `risk_model.joblib` - Risk assessment
- `loss_model.joblib` - Loss probability

Run the Flask server with: `python run.py`

In [ ]:
# Verify all models are saved
import os

models_dir = '../data/models'
for f in os.listdir(models_dir):
    path = os.path.join(models_dir, f)
    size = os.path.getsize(path) / 1024
    print(f'{f}: {size:.1f} KB')